# Dự báo nhu cầu hàng tươi sống theo ngày bằng LSTM trên FreshRetailNet-50K

Notebook này chạy thiết kế của báo cáo đề xuất bản rút gọn.

| Thí nghiệm | Mô hình | Đầu vào |
|---|---|---|
| E1 | Seasonal Naive 7 ngày (mốc tham chiếu, không học) | y(t−7) |
| E2 | LSTM | doanh số, lịch, khuyến mãi, thời tiết |
| E3 | LSTM | như E2, bỏ khuyến mãi và thời tiết |
| E4 | LSTM (phụ) | như E2, không loại ngày hết hàng khỏi hàm mất mát |

Mỗi bước xử lý đều in dữ liệu trước và sau để người đọc theo dõi. Mọi con số sinh ra khi chạy.

In [ ]:
# Colab: bỏ comment dòng dưới nếu thiếu thư viện (Colab đã có torch, pandas, pyarrow)
# !pip install -q huggingface_hub

## 0. Cấu hình

In [ ]:
N_SERIES = 1000            # số chuỗi cửa hàng × sản phẩm trong tập con
SEED = 42
STOCKOUT_MAX_HOURS = 8     # ngày có hơn 8/16 giờ hết hàng (6–22h) bị loại khỏi hàm mất mát
CAP_QUANTILE = 0.99        # chặn trên doanh số đầu vào theo phân vị 99 của từng chuỗi (ngày huấn luyện)
WINDOW = 28                # số ngày quá khứ LSTM đọc
HORIZON = 7
HIDDEN = 64
EPOCHS_MAX = 30
PATIENCE = 4
BATCH_SIZE = 256
LR = 1e-3
RUN_E4 = True              # thí nghiệm phụ: không loại ngày hết hàng
HF_REPO = "Dingdong-Inc/FreshRetailNet-50K"
HF_REVISION = "08c1fab7f9257bc73679d415d65d644165d351d4"
DATA_SOURCE = "hf"         # "hf": tải từ Hugging Face, lỗi thì dùng LOCAL_DATA_DIR; "local": chỉ dùng LOCAL_DATA_DIR
LOCAL_DATA_DIR = "data"    # chứa train.parquet, eval.parquet
OUTPUT_DIR = "outputs"
FIG_DIR = "figures"         # hình EDA và kết quả

## 1. Thư viện, seed và thời gian chạy

In [ ]:
import os, time, json, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import torch, torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything(SEED)
torch.set_num_threads(1)                      # 1 luồng để kết quả tái lập
torch.use_deterministic_algorithms(True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
T0 = time.time(); TIMES = {}
def mark(name):
    TIMES[name] = round(time.time() - T0, 1); print(f"[{name}] {TIMES[name]} s kể từ đầu")
print("numpy", np.__version__, "| pandas", pd.__version__, "| torch", torch.__version__, "| CPU threads", torch.get_num_threads())

## 2. Tải dữ liệu thô

Chỉ đọc các cột theo ngày. Hai mảng 24 giờ không dùng.

In [ ]:
DAILY_COLS = ["city_id", "store_id", "management_group_id", "product_id", "dt", "sale_amount",
              "stock_hour6_22_cnt", "discount", "holiday_flag", "activity_flag",
              "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
def locate_files():
    if DATA_SOURCE == "hf":
        try:
            from huggingface_hub import hf_hub_download
            return {s: hf_hub_download(HF_REPO, f"data/{s}.parquet", repo_type="dataset", revision=HF_REVISION)
                    for s in ("train", "eval")}, "huggingface"
        except Exception as e:
            print("Tải từ Hugging Face lỗi, dùng bản cục bộ:", repr(e)[:150])
    files = {s: os.path.join(LOCAL_DATA_DIR, f"{s}.parquet") for s in ("train", "eval")}
    assert all(os.path.exists(p) for p in files.values()), f"Không thấy dữ liệu: {files}"
    return files, "local"
FILES, SRC = locate_files()
print("Nguồn:", SRC)
train_raw = pq.read_table(FILES["train"], columns=DAILY_COLS).to_pandas()
eval_raw = pq.read_table(FILES["eval"], columns=DAILY_COLS).to_pandas()
print("Toàn bộ cột của tệp train:", pq.read_schema(FILES["train"]).names)
print("train:", train_raw.shape, "| eval:", eval_raw.shape)
print("Khoảng ngày train:", train_raw.dt.min(), "->", train_raw.dt.max(), "| eval:", eval_raw.dt.min(), "->", eval_raw.dt.max())
display(train_raw.head())
print(train_raw.dtypes.to_string())

In [ ]:
print("Giá trị thiếu theo cột (train):"); print(train_raw.isna().sum().to_string())
print("Giá trị thiếu (eval):", int(eval_raw.isna().sum().sum()))
display(train_raw[["sale_amount", "stock_hour6_22_cnt", "discount", "precpt", "avg_temperature"]].describe().round(3))
mark("load")

## 3. Chọn tập con phân tầng

Tầng = nhóm quản lý × mức hết hàng của chuỗi (ba nhóm theo tỷ lệ giờ hết hàng trên tập train). Phân bổ theo tỷ lệ, seed cố định.

In [ ]:
ser = (train_raw.groupby(["store_id", "product_id"])
       .agg(mg=("management_group_id", "first"), city=("city_id", "first"),
            so_rate=("stock_hour6_22_cnt", lambda s: s.mean() / 16)).reset_index())
ser["so_level"] = pd.qcut(ser["so_rate"].rank(method="first"), 3, labels=["thấp", "vừa", "cao"])
ser["stratum"] = ser["mg"].astype(str) + "_" + ser["so_level"].astype(str)
print("Trước: số chuỗi =", len(ser))
rng = np.random.default_rng(SEED)
alloc = (ser["stratum"].value_counts() / len(ser) * N_SERIES)
n_take = np.floor(alloc).astype(int)
rest = N_SERIES - n_take.sum()
n_take[(alloc - np.floor(alloc)).sort_values(ascending=False).index[:rest]] += 1
chosen = pd.concat([g.iloc[rng.permutation(len(g))[:n_take[s]]] for s, g in ser.groupby("stratum")])
chosen = chosen.sort_values(["store_id", "product_id"]).reset_index(drop=True)
print("Sau: số chuỗi =", len(chosen))
comp = pd.DataFrame({"toàn bộ (%)": ser["stratum"].value_counts(normalize=True) * 100,
                     "tập con (%)": chosen["stratum"].value_counts(normalize=True) * 100}).round(2)
display(comp.sort_index())
print("Tỷ lệ giờ hết hàng trung bình: toàn bộ %.3f | tập con %.3f" % (ser.so_rate.mean(), chosen.so_rate.mean()))

In [ ]:
keys = chosen[["store_id", "product_id"]]
df = pd.concat([train_raw.assign(split_src="train"), eval_raw.assign(split_src="eval")])
print("Trước khi lọc:", df.shape)
df = df.merge(keys, on=["store_id", "product_id"], how="inner")
df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)
print("Sau khi lọc tập con:", df.shape, "=", len(keys), "chuỗi ×", df.groupby(["store_id", "product_id"]).size().iloc[0], "ngày")
assert (df.groupby(["store_id", "product_id"]).size() == 97).all()
display(df.head(3)); display(df.tail(3))
del train_raw, eval_raw
mark("subset")

## 4. Bảng theo ngày, thêm thứ trong tuần và chỉ số ngày

Ngày 0–82: huấn luyện. Ngày 83–89: validation. Ngày 90–96: kiểm tra (tập eval).

In [ ]:
print("Cột trước:", list(df.columns))
df["dt"] = pd.to_datetime(df["dt"])
days = np.sort(df["dt"].unique())
df["day"] = np.searchsorted(days, df["dt"].values)
df["dow"] = df["dt"].dt.dayofweek          # 0 = thứ Hai
df["split"] = np.where(df.day <= 82, "train", np.where(df.day <= 89, "val", "test"))
print("Cột sau:", list(df.columns))
print(df.groupby("split").agg(so_dong=("day", "size"), tu_ngay=("dt", "min"), den_ngay=("dt", "max")))
display(df[["store_id", "product_id", "dt", "day", "dow", "split", "sale_amount", "holiday_flag"]].head(8))
print("holiday_flag theo thứ trong tuần (tỷ lệ = 1):"); print(df.groupby("dow")["holiday_flag"].mean().round(3).to_string())

## 5. Khám phá dữ liệu (EDA)

Phân tích trên tập con 1,000 chuỗi với doanh số gốc, trước mọi bước sửa. Các ô sau in số liệu và lưu hình vào thư mục figures.

In [ ]:
s = df.sale_amount
print("Số chuỗi – ngày:", len(s))
print("Tỷ lệ ngày doanh số = 0: %.2f%%" % ((s == 0).mean() * 100))
print("Độ lệch (skewness): %.2f" % s.skew())
print("Phân vị:"); print(s.quantile([0.25, 0.5, 0.75, 0.9, 0.99, 1.0]).round(3).to_string())
daily = df.groupby("dt")["sale_amount"].sum()
wk = df[df.split == "train"].groupby("dow")["sale_amount"].mean()
print("Doanh số trung bình theo thứ (ngày huấn luyện, 0 = thứ Hai):"); print(wk.round(3).to_string())
print("Thứ cao nhất / thấp nhất: %.2f lần" % (wk.max() / wk.min()))
hol = df[df.split == "train"].groupby("holiday_flag")["sale_amount"].mean()
print("Doanh số trung bình theo holiday_flag:", hol.round(3).to_dict())
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(np.log1p(s), bins=60, color="#2874A6"); ax[0].set_title("Phân bố log(1 + sale_amount)"); ax[0].set_xlabel("log(1 + sale_amount)")
ax[1].plot(daily.index, daily.values, color="#1A5276")
for d_, lab, ha in [(days[83], "val ", "right"), (days[90], " test", "left")]:
    ax[1].axvline(d_, color="gray", ls="--"); ax[1].text(d_, daily.max(), lab, va="top", ha=ha)
ax[1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2)); ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax[1].set_title("Tổng doanh số theo ngày (1,000 chuỗi)"); ax[1].tick_params(axis="x", rotation=30)
ax[2].bar(["T2", "T3", "T4", "T5", "T6", "T7", "CN"], wk.values, color="#2874A6"); ax[2].set_title("Doanh số trung bình theo thứ (train)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/eda_sales.png", dpi=150); plt.show()

In [ ]:
def spearman(a, b): return a.rank().corr(b.rank())   # tương quan hạng, không cần scipy
tr = df[df.split == "train"]
so = tr.groupby("stock_hour6_22_cnt")["sale_amount"].agg(["size", "mean"])
so["share_%"] = so["size"] / so["size"].sum() * 100
print("Số giờ hết hàng 6–22h: tỷ lệ ngày và doanh số trung bình (train):"); print(so.round(3).to_string())
print("Hệ số tương quan Spearman giữa số giờ hết hàng và doanh số: %.3f" % spearman(tr.stock_hour6_22_cnt, tr.sale_amount))
pr = tr.groupby("activity_flag")["sale_amount"].agg(["size", "mean"]); print("Theo activity_flag:"); print(pr.round(3).to_string())
tr_d = tr.assign(disc_bin=pd.cut(tr.discount, [-0.01, 0.7, 0.85, 0.95, 0.999, 1.1], labels=["<=0.70", "0.70-0.85", "0.85-0.95", "0.95-1", "=1 (không giảm)"]))
db = tr_d.groupby("disc_bin", observed=True)["sale_amount"].agg(["size", "mean"]); print("Theo mức discount:"); print(db.round(3).to_string())
rain = tr.assign(rain=np.where(tr.precpt > 0, "có mưa", "không mưa")).groupby("rain")["sale_amount"].agg(["size", "mean"])
print("Theo mưa:"); print(rain.round(3).to_string())
tq = tr.assign(tq=pd.qcut(tr.avg_temperature, 5)).groupby("tq", observed=True)["sale_amount"].mean()
print("Theo ngũ phân vị nhiệt độ:"); print(tq.round(3).to_string())
for c in ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]:
    print(f"Spearman({c}, sale_amount) = {spearman(tr[c], tr.sale_amount):.3f}")
mg = df[df.split == "train"].groupby("management_group_id").agg(
    so_chuoi=("product_id", lambda x: x.size // 83), dt_tb=("sale_amount", "mean"),
    ty_le_0=("sale_amount", lambda x: (x == 0).mean()), gio_het_hang_tb=("stock_hour6_22_cnt", "mean"))
print("Theo nhóm quản lý (train):"); print(mg.round(3).to_string())
fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
ax[0].bar(so.index, so["mean"], color="#2874A6"); ax[0].set_title("Doanh số TB theo số giờ hết hàng"); ax[0].set_xlabel("số giờ hết hàng 6–22h")
ax[1].bar(["không KM", "có KM"], pr["mean"].values, color=["#85929E", "#1A5276"]); ax[1].set_title("Doanh số TB theo activity_flag")
ax[2].plot(range(5), tq.values, marker="o", color="#1A5276"); ax[2].set_xticks(range(5)); ax[2].set_xticklabels(["Q1", "Q2", "Q3", "Q4", "Q5"])
ax[2].set_title("Doanh số TB theo ngũ phân vị nhiệt độ")
ax[3].bar(mg.index.astype(str), mg["dt_tb"], color="#2874A6"); ax[3].set_title("Doanh số TB theo nhóm quản lý"); ax[3].set_xlabel("management_group_id")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/eda_drivers.png", dpi=150); plt.show()
mark("eda")

## 6. Sửa discount

discount là tỷ lệ giá (1.0 = không giảm). Giá trị 0 được coi là thiếu và điền bằng trung vị của sản phẩm trong cùng trạng thái activity_flag (tính trên ngày huấn luyện). Giá trị lớn hơn 1 được cắt về 1.

In [ ]:
print("Trước: discount == 0:", int((df.discount == 0).sum()), "| discount > 1:", int((df.discount > 1).sum()))
display(df.loc[df.discount == 0, ["store_id", "product_id", "dt", "discount", "activity_flag"]].head(5))
ok = (df.discount > 0) & (df.split == "train")
med_pa = df[ok].groupby(["product_id", "activity_flag"])["discount"].median()
med_a = df[ok].groupby("activity_flag")["discount"].median()
z = df.discount == 0
fill = df.loc[z].set_index(["product_id", "activity_flag"]).index.map(med_pa.to_dict().get)
fill = pd.Series(fill, index=df.index[z]).astype(float)
fill = fill.fillna(df.loc[z, "activity_flag"].map(med_a))
idx_z = df.index[z]
df.loc[z, "discount"] = fill.values
df["discount"] = df["discount"].clip(upper=1.0)
print("Sau: discount == 0:", int((df.discount == 0).sum()), "| discount > 1:", int((df.discount > 1).sum()))
display(df.loc[idx_z[:5], ["store_id", "product_id", "dt", "discount", "activity_flag"]])
print(df.discount.describe().round(3).to_string())

## 7. Quy tắc hết hàng

Ngày có stock_hour6_22_cnt > 8 (hết hàng hơn nửa số giờ 6–22h) bị loại khỏi hàm mất mát và khỏi đánh giá chính. Dòng vẫn được giữ làm đầu vào.

In [ ]:
df["censored"] = df.stock_hour6_22_cnt > STOCKOUT_MAX_HOURS
print("Phân bố số giờ hết hàng (6–22h), toàn tập con:")
print(df.stock_hour6_22_cnt.value_counts().sort_index().to_string())
tab = df.groupby("split").agg(so_ngay=("censored", "size"), bi_loai=("censored", "sum"))
tab["ty_le_loai_%"] = (tab.bi_loai / tab.so_ngay * 100).round(2)
display(tab)
print("Doanh số trung bình: ngày bị loại %.3f | ngày giữ lại %.3f" % (df[df.censored].sale_amount.mean(), df[~df.censored].sale_amount.mean()))
display(df.loc[df.censored, ["store_id", "product_id", "dt", "sale_amount", "stock_hour6_22_cnt", "censored"]].head(5))

## 8. Ngoại lai của sale_amount

Doanh số đầu vào của LSTM được chặn trên theo phân vị 99 của từng chuỗi, tính trên 83 ngày huấn luyện. Mục tiêu dự báo và giá trị dùng để đánh giá giữ nguyên.

In [ ]:
print("Trước:"); print(df.sale_amount.describe().round(3).to_string())
p99 = df[df.split == "train"].groupby(["store_id", "product_id"])["sale_amount"].quantile(CAP_QUANTILE).rename("p99")
df = df.merge(p99.reset_index(), on=["store_id", "product_id"], how="left")
df["sale_in"] = np.minimum(df.sale_amount, df.p99)
capped = df.sale_amount > df.p99
print("Số giá trị bị chặn: %d / %d (%.2f%%); số chuỗi có ít nhất một giá trị bị chặn: %d"
      % (capped.sum(), len(df), capped.mean() * 100, df[capped].groupby(["store_id", "product_id"]).ngroups))
print("Theo tập:"); print(capped.groupby(df.split).sum().to_string())
print("Sau (sale_in):"); print(df.sale_in.describe().round(3).to_string())
display(df.loc[capped, ["store_id", "product_id", "dt", "sale_amount", "p99", "sale_in"]].head(5))
mark("preprocess")

## 9. Đặc trưng

Mỗi ngày có 9 biến đầu vào gốc: doanh số, thứ trong tuần, holiday_flag, discount, activity_flag và 4 biến thời tiết. Thứ trong tuần được mã hóa one-hot (7 cột). Trường hợp E3 bỏ discount, activity_flag và thời tiết.

In [ ]:
WEATHER = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
PROMO = ["discount", "activity_flag"]
DOW_COLS = [f"dow_{k}" for k in range(7)]
print("Cột trước:", df.shape[1])
for k in range(7): df[f"dow_{k}"] = (df.dow == k).astype(np.float32)
print("Cột sau:", df.shape[1], "| thêm:", DOW_COLS)
display(df[["dt", "dow"] + DOW_COLS + ["holiday_flag"] + PROMO + WEATHER].head(7))

## 10. Chuẩn hóa

Doanh số chia cho trung bình của chuỗi trên ngày huấn luyện (sau khi chặn). Thời tiết chuẩn hóa z-score bằng trung bình và độ lệch chuẩn của ngày huấn luyện. discount nằm trong [0, 1] nên giữ nguyên.

In [ ]:
print("Trước:"); display(df[["sale_in"] + WEATHER].describe().round(3))
scale = df[df.split == "train"].groupby(["store_id", "product_id"])["sale_in"].mean().clip(lower=0.01).rename("scale")
df = df.merge(scale.reset_index(), on=["store_id", "product_id"], how="left")
df["sale_n"] = df.sale_in / df.scale
mu = df.loc[df.split == "train", WEATHER].mean(); sd = df.loc[df.split == "train", WEATHER].std()
for c in WEATHER: df[c + "_z"] = (df[c] - mu[c]) / sd[c]
WEATHER_Z = [c + "_z" for c in WEATHER]
print("Sau:"); display(df[["sale_n"] + WEATHER_Z].describe().round(3))
print("Giá trị thiếu sau xử lý:", int(df[["sale_n", "discount"] + WEATHER_Z].isna().sum().sum()))

## 11. Tạo cửa sổ cho LSTM

Mỗi mẫu: 28 ngày quá khứ đến mốc T, đầu ra 7 ngày T+1 … T+7. Biến biết trước của 7 ngày tới đi vào nhánh riêng.
- Huấn luyện: T = 27 … 75 (49 mẫu mỗi chuỗi).
- Validation: T = 82. Kiểm tra: T = 89.
- Huấn luyện lại sau khi chọn số epoch: T = 27 … 82 (56 mẫu mỗi chuỗi).

In [ ]:
df = df.sort_values(["store_id", "product_id", "day"]).reset_index(drop=True)
N = df.groupby(["store_id", "product_id"]).ngroups; D = 97
def arr(c): return df[c].to_numpy(np.float32).reshape(N, D)
def build_inputs(use_promo_weather):
    past_cols = ["sale_n"] + DOW_COLS + ["holiday_flag"] + ((PROMO + WEATHER_Z) if use_promo_weather else [])
    fut_cols = DOW_COLS + ["holiday_flag"] + ((PROMO + WEATHER_Z) if use_promo_weather else [])
    P = np.stack([arr(c) for c in past_cols], -1)     # N × 97 × số biến quá khứ
    F = np.stack([arr(c) for c in fut_cols], -1)      # N × 97 × số biến tương lai
    return P, F, past_cols, fut_cols
Y = arr("sale_amount"); YN = Y / arr("scale"); SCALE = arr("scale")[:, 0]
KEEP = (~df["censored"].to_numpy().reshape(N, D)).astype(np.float32)
PROMO_DAY = arr("activity_flag")
def windows(P, F, origins, keep):
    i = np.repeat(np.arange(N), len(origins)); t = np.tile(origins, N)
    past = np.stack([P[i, t - WINDOW + 1 + k] for k in range(WINDOW)], 1)
    fut = np.stack([F[i, t + 1 + k] for k in range(HORIZON)], 1)
    y = np.stack([YN[i, t + 1 + k] for k in range(HORIZON)], 1)
    m = np.stack([keep[i, t + 1 + k] for k in range(HORIZON)], 1)
    return past, fut, y, m, i
P2, F2, past_cols, fut_cols = build_inputs(True)
print("Biến quá khứ (%d):" % len(past_cols), past_cols)
print("Biến biết trước (%d):" % len(fut_cols), fut_cols)
TR_ORIG = np.arange(WINDOW - 1, 83 - HORIZON); FULL_ORIG = np.arange(WINDOW - 1, 90 - HORIZON)
for name, o in [("train", TR_ORIG), ("val", np.array([82])), ("test", np.array([89])), ("refit", FULL_ORIG)]:
    xp, xf, y, m, _ = windows(P2, F2, o, KEEP)
    print(f"{name:5s}: mốc {o.min()}–{o.max()} | mẫu {xp.shape[0]:,} | X_past {xp.shape} | X_fut {xf.shape} | y {y.shape} | ngày mục tiêu bị loại {int((m == 0).sum()):,}/{m.size:,}")
xp, xf, y, m, _ = windows(P2, F2, np.array([82]), KEEP)
print("Ví dụ mẫu validation của chuỗi 0, 5 ngày cuối của cửa sổ quá khứ:")
display(pd.DataFrame(xp[0, -5:], columns=past_cols).round(3))
print("Mục tiêu (đã chuẩn hóa):", y[0].round(3), "| mặt nạ:", m[0])
mark("features_windows")

## 12. Mô hình LSTM

Một lớp LSTM, 64 đơn vị ẩn, đọc cửa sổ 28 ngày. Trạng thái ẩn cuối ghép với biến biết trước của 7 ngày tới, qua lớp kết nối đầy đủ, ra 7 giá trị. Hàm mất mát MSE có mặt nạ. Early stopping theo WAPE validation, sau đó huấn luyện lại trên 90 ngày với số epoch tốt nhất.

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_past, n_fut):
        super().__init__()
        self.lstm = nn.LSTM(n_past, HIDDEN, batch_first=True)
        self.head = nn.Sequential(nn.Linear(HIDDEN + HORIZON * n_fut, 128), nn.ReLU(), nn.Linear(128, HORIZON))
    def forward(self, xp, xf):
        _, (h, _) = self.lstm(xp)
        return self.head(torch.cat([h[-1], xf.flatten(1)], 1))

def wape(pred, true, mask=None):
    if mask is None: mask = np.ones_like(true, bool)
    return float(np.abs(pred - true)[mask].sum() / true[mask].sum())

def predict(model, P, F, origin):
    xp, xf, _, _, i = windows(P, F, np.array([origin]), KEEP)
    model.eval()
    with torch.no_grad():
        out = model(torch.from_numpy(xp), torch.from_numpy(xf)).numpy()
    return np.clip(out * SCALE[i][:, None], 0, None)          # về đơn vị gốc

def fit(P, F, origins, keep, epochs, val=True):
    seed_everything(SEED)
    xp, xf, y, m, _ = windows(P, F, origins, keep)
    model = LSTMForecaster(xp.shape[2], xf.shape[2]); opt = torch.optim.Adam(model.parameters(), lr=LR)
    T = [torch.from_numpy(a) for a in (xp, xf, y, m)]
    g = torch.Generator().manual_seed(SEED)
    best, best_ep, bad, hist = np.inf, 0, 0, []
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(len(xp), generator=g); tl = 0.0
        for b in range(0, len(xp), BATCH_SIZE):
            j = perm[b:b + BATCH_SIZE]
            out = model(T[0][j], T[1][j])
            loss = ((out - T[2][j]) ** 2 * T[3][j]).sum() / T[3][j].sum().clamp(min=1)
            opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item() * len(j)
        rec = {"epoch": ep, "train_loss": tl / len(xp)}
        if val:
            pv = predict(model, P, F, 82); yv = Y[:, 83:90]; mv = KEEP[:, 83:90] > 0
            rec["val_wape"] = wape(pv, yv, mv)
            if rec["val_wape"] < best - 1e-5: best, best_ep, bad = rec["val_wape"], ep, 0
            else: bad += 1
        hist.append(rec)
        print("  " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        if val and bad >= PATIENCE: break
    return model, best_ep, pd.DataFrame(hist)

def run_lstm(name, use_promo_weather, keep):
    t = time.time()
    P, F, pc, fc = build_inputs(use_promo_weather)
    print(f"{name}: {len(pc)} biến quá khứ, {len(fc)} biến biết trước")
    _, best_ep, hist = fit(P, F, TR_ORIG, keep, EPOCHS_MAX, val=True)
    print(f"{name}: epoch tốt nhất = {best_ep}; huấn luyện lại trên 90 ngày")
    model, _, _ = fit(P, F, FULL_ORIG, keep, best_ep, val=False)
    pred = predict(model, P, F, 89)
    hist.to_csv(f"{OUTPUT_DIR}/history_{name}.csv", index=False)
    INFO[name] = {"best_epoch": best_ep, "n_params": sum(p.numel() for p in model.parameters()), "sec": round(time.time() - t, 1)}
    print(name, INFO[name])
    return pred
INFO, PRED = {}, {}

## 13. E1: Seasonal Naive 7 ngày

In [ ]:
PRED["E1"] = Y[:, 83:90].copy()       # ŷ(t) = y(t−7) cho t = 90 … 96
print("Dự báo E1, 3 chuỗi đầu:"); print(PRED["E1"][:3].round(3))

## 14. E2: LSTM đầy đủ đặc trưng

In [ ]:
PRED["E2"] = run_lstm("E2", True, KEEP)
mark("E2")

## 15. E3: LSTM bỏ khuyến mãi và thời tiết

In [ ]:
PRED["E3"] = run_lstm("E3", False, KEEP)
mark("E3")

## 16. E4 (phụ): LSTM không loại ngày hết hàng khỏi hàm mất mát

In [ ]:
if RUN_E4:
    PRED["E4"] = run_lstm("E4", True, np.ones_like(KEEP))
    mark("E4")

## 17. Đánh giá

Đánh giá chính trên các chuỗi – ngày kiểm tra có tối đa 8 giờ hết hàng. Kết quả trên toàn bộ 7 ngày kiểm tra in kèm. Doanh số và dự báo ở đơn vị chuẩn hóa của bộ dữ liệu.

In [ ]:
YT = Y[:, 90:97]; MT = KEEP[:, 90:97] > 0; ALL = np.ones_like(MT)
def metrics(p, mask):
    e = (p - YT)[mask]
    return {"WAPE": float(np.abs(e).sum() / YT[mask].sum()), "MAE": float(np.abs(e).mean()), "RMSE": float(np.sqrt((e ** 2).mean()))}
rows = []
for k, p in PRED.items():
    r = {"exp": k, **metrics(p, MT)}
    r.update({f"{a}_all": b for a, b in metrics(p, ALL).items()})
    rows.append(r)
RES = pd.DataFrame(rows).set_index("exp")
for k in RES.index:
    if k != "E1": RES.loc[k, "WAPE_giam_vs_E1_%"] = (1 - RES.loc[k, "WAPE"] / RES.loc["E1", "WAPE"]) * 100
print("Số chuỗi – ngày kiểm tra được đánh giá:", int(MT.sum()), "/", MT.size)
display(RES.round(4)); RES.to_csv(f"{OUTPUT_DIR}/results.csv")

In [ ]:
hz = pd.DataFrame({k: [wape(p[:, h], YT[:, h], MT[:, h]) for h in range(7)] for k, p in PRED.items()}, index=pd.Index(range(1, 8), name="h"))
promo = PROMO_DAY[:, 90:97] > 0
grp = pd.DataFrame({k: {"ngày có khuyến mãi": wape(p, YT, MT & promo), "ngày không khuyến mãi": wape(p, YT, MT & ~promo)} for k, p in PRED.items()})
grp["số chuỗi – ngày"] = [int((MT & promo).sum()), int((MT & ~promo).sum())]
print("WAPE theo tầm dự báo:"); display(hz.round(4))
print("WAPE theo khuyến mãi:"); display(grp.round(4))
hz.to_csv(f"{OUTPUT_DIR}/wape_by_horizon.csv"); grp.to_csv(f"{OUTPUT_DIR}/wape_by_promo.csv")

## 18. Biểu đồ

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k in hz.columns: ax[0].plot(hz.index, hz[k], marker="o", label=k)
ax[0].set_xlabel("tầm dự báo h (ngày)"); ax[0].set_ylabel("WAPE"); ax[0].set_title("WAPE theo tầm dự báo"); ax[0].legend()
tot = Y.sum(0); x = pd.to_datetime(days)
ax[1].plot(x[60:], tot[60:], color="black", label="thực tế")
for k in ["E1", "E2"]: ax[1].plot(x[90:], PRED[k].sum(0), marker="o", label=k)
ax[1].axvline(x[89], color="gray", ls="--"); ax[1].set_title("Tổng doanh số 1,000 chuỗi: thực tế và dự báo"); ax[1].legend()
ax[1].xaxis.set_major_locator(mdates.DayLocator(interval=5)); ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/results_forecast.png", dpi=150); plt.show()

## 19. Tóm tắt lần chạy

In [ ]:
summary = {"n_series": int(N), "data_source": SRC, "seed": SEED, "torch_threads": torch.get_num_threads(),
           "times_sec": TIMES, "total_sec": round(time.time() - T0, 1), "lstm": INFO,
           "results": RES.round(6).reset_index().to_dict(orient="records")}
json.dump(summary, open(f"{OUTPUT_DIR}/run_summary.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps({k: summary[k] for k in ["n_series", "times_sec", "total_sec", "lstm"]}, ensure_ascii=False, indent=1))